# Argus VLM Optimization — Notebook 02: Long Context KV Cache Benchmark

**Goal:** Evaluate KV-cache memory scaling and latency over multi-frame synthetic contexts (1K, 5K, 10K, 20K, 50K tokens).


In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score
!pip install -q hqq optimum quanto || true


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import pandas as pd
import yaml
from PIL import Image

from src.vlm.qwen_vlm import QwenVLMWrapper
from src.kv_cache.benchmark import KVCacheConfig
from src.benchmarking.benchmark_runner import append_csv, clear_gpu, measure_peak_vram_gb, reset_vram_peak
from src.visualization.plots import plot_vram_vs_context, plot_latency_vs_context

print(f"CUDA: {torch.cuda.is_available()}")


In [ ]:
# Cell 3: Configuration
context_lengths = [1000, 5000, 10000, 20000, 50000]
configs_to_test = [
    KVCacheConfig(name="baseline_fp16", backend="dynamic", nbits=16),
    KVCacheConfig(name="quantized_hqq_4bit", backend="HQQ", nbits=4),
    KVCacheConfig(name="quantized_quanto_4bit", backend="quanto", nbits=4),
]


In [ ]:
# Cell 4: Model Loading
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
try:
    wrapper = QwenVLMWrapper(model_name=model_name, device_map="auto" if torch.cuda.is_available() else "cpu")
except Exception as e:
    print(f"Model load notice: {e}")
    wrapper = None


In [ ]:
# Cell 5: Synthetic Long-Context Input Generation
def create_synthetic_context(token_length: int) -> str:
    base_text = "Camera 01: Hallway monitoring. Normal background motion detected. Person moving west. "
    repetitions = (token_length // 12) + 1
    return (base_text * repetitions)[:token_length * 4]

sample_img = Image.new("RGB", (224, 224), color=(100, 100, 100))


In [ ]:
# Cell 6: Long Context Benchmark Sweep
results_list = []
output_csv = repo_root / "results" / "kv_cache" / "long_context_results.csv"

for ctx_len in context_lengths:
    long_prompt = create_synthetic_context(ctx_len)
    for cfg in configs_to_test:
        clear_gpu()
        reset_vram_peak()
        if wrapper is None or not torch.cuda.is_available():
            row = {
                "context_tokens": ctx_len,
                "config": cfg.name,
                "backend": cfg.backend,
                "nbits": cfg.nbits,
                "peak_memory_gb": 0.0,
                "latency_seconds": 0.0,
                "tokens_per_second": 0.0,
                "status": "FAILED",
                "error": "CUDA or Model unavailable"
            }
        else:
            try:
                res = wrapper.generate(
                    image=sample_img,
                    prompt=long_prompt,
                    max_new_tokens=32,
                    kv_cache_config=cfg.to_kv_config_dict()
                )
                row = {
                    "context_tokens": ctx_len,
                    "config": cfg.name,
                    "backend": cfg.backend,
                    "nbits": cfg.nbits,
                    "peak_memory_gb": res.peak_vram_gb,
                    "latency_seconds": res.latency_seconds,
                    "tokens_per_second": res.tokens_per_second,
                    "status": res.status,
                    "error": res.error
                }
            except Exception as e:
                row = {
                    "context_tokens": ctx_len,
                    "config": cfg.name,
                    "backend": cfg.backend,
                    "nbits": cfg.nbits,
                    "peak_memory_gb": 0.0,
                    "latency_seconds": 0.0,
                    "tokens_per_second": 0.0,
                    "status": "FAILED",
                    "error": str(e)
                }
        append_csv(row, output_csv)
        results_list.append(row)


In [ ]:
# Cell 7: Plotting & Results Analysis
df = pd.DataFrame(results_list)
print(df)
fig_dir = repo_root / "results" / "figures"
plot_vram_vs_context(df, output_path=str(fig_dir / "kv_vram_vs_context.png"))
plot_latency_vs_context(df, output_path=str(fig_dir / "kv_latency_vs_context.png"))
print("Plots generated in results/figures/")
